# AGENTS026 – Hours 3–4: Embedding Service & Vector Store

This notebook adds semantic retrieval to the AGENTS026 workflow. It loads previously created incident candidates, builds a GPU-capable embedding pipeline with ROCm PyTorch-compatible libraries, implements `embed_incident(incident)`, and sets up a FAISS index for similarity search over historical incidents and runbooks.

The goal for this hour is not perfect retrieval quality; it is to establish a robust retrieval layer that later RCA agents can call.


## Section 1 – Install / Import Dependencies

We use `sentence-transformers` for embeddings and `faiss-cpu` for the vector index. In many AMD notebook images, PyTorch is already present with ROCm support.


In [1]:
%pip install -q sentence-transformers faiss-cpu pandas numpy

print('Installed / ensured sentence-transformers, faiss-cpu, pandas, numpy')


Note: you may need to restart the kernel to use updated packages.
Installed / ensured sentence-transformers, faiss-cpu, pandas, numpy


In [1]:
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any
import json

import numpy as np
import pandas as pd
import torch
import faiss
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer

print('PyTorch version:', torch.__version__)
print('CUDA/HIP available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))


PyTorch version: 2.12.0+cu130
CUDA/HIP available: False


## Section 2 – Paths and Inputs

Load incident candidates produced in the Hours 2–3 notebook.


In [2]:
root = Path.cwd() / 'agents026'
data_dir = root / 'data'
incidents_dir = data_dir / 'incidents'
vector_dir = data_dir / 'vectorstore'
vector_dir.mkdir(parents=True, exist_ok=True)

incident_candidates_path = incidents_dir / 'incident_candidates.json'
with open(incident_candidates_path, 'r', encoding='utf-8') as f:
    incident_candidates_raw = json.load(f)

len(incident_candidates_raw), incident_candidates_raw[0] if incident_candidates_raw else 'No incidents found'


(4,
 {'incident_id': 'inc-001',
  'start_time': '2026-06-10 12:19:00',
  'end_time': '2026-06-10 12:19:00',
  'services': ['catalog-api'],
  'anomaly_type': 'error_rate',
  'metric_summary': {'cpu_max': 26.1119737852507,
   'latency_p95_max': 161.08106122719428,
   'error_rate_max': 0.0032061994146254,
   'rps_avg': 30.86440651983976},
  'log_samples': ['2026-06-10T12:17:00 [INFO] catalog-api handled request batch successfully',
   '2026-06-10T12:18:00 [INFO] catalog-api handled request batch successfully',
   '2026-06-10T12:19:00 [INFO] catalog-api handled request batch successfully',
   '2026-06-10T12:20:00 [INFO] catalog-api handled request batch successfully',
   '2026-06-10T12:21:00 [INFO] catalog-api handled request batch successfully',
   '2026-06-10T12:22:00 [INFO] catalog-api handled request batch successfully',
   '2026-06-10T12:23:00 [INFO] catalog-api handled request batch successfully',
   '2026-06-10T12:24:00 [INFO] catalog-api handled request batch successfully'],
  'k8s

## Section 3 – Recreate Core Schemas

Define the structured objects we will embed and retrieve.


In [3]:
class IncidentCandidate(BaseModel):
    incident_id: str
    start_time: datetime
    end_time: datetime
    services: List[str]
    anomaly_type: str
    metric_summary: Dict[str, float] = Field(default_factory=dict)
    log_samples: List[str] = Field(default_factory=list)
    k8s_event_samples: List[str] = Field(default_factory=list)
    change_refs: List[str] = Field(default_factory=list)

incident_candidates = [IncidentCandidate(**x) for x in incident_candidates_raw]
incident_candidates[:2]


[IncidentCandidate(incident_id='inc-001', start_time=datetime.datetime(2026, 6, 10, 12, 19), end_time=datetime.datetime(2026, 6, 10, 12, 19), services=['catalog-api'], anomaly_type='error_rate', metric_summary={'cpu_max': 26.1119737852507, 'latency_p95_max': 161.08106122719428, 'error_rate_max': 0.0032061994146254, 'rps_avg': 30.86440651983976}, log_samples=['2026-06-10T12:17:00 [INFO] catalog-api handled request batch successfully', '2026-06-10T12:18:00 [INFO] catalog-api handled request batch successfully', '2026-06-10T12:19:00 [INFO] catalog-api handled request batch successfully', '2026-06-10T12:20:00 [INFO] catalog-api handled request batch successfully', '2026-06-10T12:21:00 [INFO] catalog-api handled request batch successfully', '2026-06-10T12:22:00 [INFO] catalog-api handled request batch successfully', '2026-06-10T12:23:00 [INFO] catalog-api handled request batch successfully', '2026-06-10T12:24:00 [INFO] catalog-api handled request batch successfully'], k8s_event_samples=[], 

## Section 4 – Create Synthetic Historical Incidents & Runbooks

In addition to current incident candidates, we create a few synthetic historical incident summaries and runbooks. These form the first version of our retrieval corpus.


In [4]:
historical_incidents = [
    {
        'doc_id': 'hist-001',
        'doc_type': 'historical_incident',
        'service': 'checkout-api',
        'title': 'Checkout latency spike after deployment',
        'text': 'checkout-api experienced elevated latency and intermittent HTTP 500 responses after deployment v3.2.1. Root cause was a slow downstream dependency call amplified by aggressive retry settings. Mitigation involved rollback and retry tuning.',
    },
    {
        'doc_id': 'hist-002',
        'doc_type': 'historical_incident',
        'service': 'payments-service',
        'title': 'Payment error spike due to DB pool exhaustion',
        'text': 'payments-service showed high error rate because the database connection pool was exhausted during peak request volume. Pod restarts gave temporary relief. Permanent fix was increasing pool size and reducing long-running transactions.',
    },
    {
        'doc_id': 'hist-003',
        'doc_type': 'historical_incident',
        'service': 'auth-service',
        'title': 'Auth failures after config change',
        'text': 'auth-service began rejecting requests after a configuration update changed token validation settings. Symptoms included increased 401 responses and elevated CPU from repeated retries. The fix was reverting the config and flushing stale caches.',
    },
]

runbooks = [
    {
        'doc_id': 'rb-001',
        'doc_type': 'runbook',
        'service': 'generic',
        'title': 'Investigate latency regression after deploy',
        'text': 'Check recent deployment and config changes. Compare pre-deploy and post-deploy latency. Inspect downstream dependency timings, rollback if needed, and validate after mitigation.',
    },
    {
        'doc_id': 'rb-002',
        'doc_type': 'runbook',
        'service': 'generic',
        'title': 'Database connection pool exhaustion remediation',
        'text': 'Inspect active DB connections, queue depth, and slow queries. Temporarily scale replicas, restart only unhealthy pods if required, increase pool size carefully, and identify traffic spikes or query regressions.',
    },
    {
        'doc_id': 'rb-003',
        'doc_type': 'runbook',
        'service': 'generic',
        'title': 'Rollback bad configuration change',
        'text': 'Identify the latest config change, confirm blast radius, roll back to the last known good config, restart affected services if necessary, and monitor errors and latency for recovery.',
    },
]

len(historical_incidents), len(runbooks)


(3, 3)

## Section 5 – Convert Incidents to Text for Embedding

Embedding quality usually improves if we create a compact but information-dense text representation.


In [5]:
def incident_to_text(incident: IncidentCandidate) -> str:
    metric_bits = ', '.join([f'{k}={v:.4f}' for k, v in incident.metric_summary.items()])
    logs = ' | '.join(incident.log_samples[:5])
    k8s = ' | '.join(incident.k8s_event_samples[:3])
    changes = ' | '.join(incident.change_refs[:3])
    return (
        f'incident_id={incident.incident_id}; '
        f'services={','.join(incident.services)}; '
        f'anomaly_type={incident.anomaly_type}; '
        f'start_time={incident.start_time.isoformat()}; '
        f'end_time={incident.end_time.isoformat()}; '
        f'metrics=[{metric_bits}]; '
        f'logs=[{logs}]; '
        f'k8s_events=[{k8s}]; '
        f'changes=[{changes}]'
    )

incident_texts = [incident_to_text(x) for x in incident_candidates]
incident_texts[0][:1200] if incident_texts else 'No incident text available'


'incident_id=inc-001; services=catalog-api; anomaly_type=error_rate; start_time=2026-06-10T12:19:00; end_time=2026-06-10T12:19:00; metrics=[cpu_max=26.1120, latency_p95_max=161.0811, error_rate_max=0.0032, rps_avg=30.8644]; logs=[2026-06-10T12:17:00 [INFO] catalog-api handled request batch successfully | 2026-06-10T12:18:00 [INFO] catalog-api handled request batch successfully | 2026-06-10T12:19:00 [INFO] catalog-api handled request batch successfully | 2026-06-10T12:20:00 [INFO] catalog-api handled request batch successfully | 2026-06-10T12:21:00 [INFO] catalog-api handled request batch successfully]; k8s_events=[]; changes=[2026-06-10T12:13:00 [feature_flag] feature_flag applied to catalog-api at 2026-06-10T12:13:00 (v2.2.2) | 2026-06-10T12:16:00 [deploy] deploy applied to catalog-api at 2026-06-10T12:16:00 (v2.1.8)]'

## Section 6 – Load Embedding Model on GPU (if available)

A compact sentence-transformer model is usually enough here. If ROCm GPU is available, we push the model to `cuda`; otherwise this notebook still works on CPU.


In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print('Embedding model:', EMBED_MODEL_NAME)
print('Embedding device:', device)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding device: cpu


## Section 7 – GPU-Accelerated Embedding Functions

Implement reusable helpers for embedding incidents and arbitrary documents.


In [7]:
def embed_texts(texts: List[str], batch_size: int = 16) -> np.ndarray:
    vectors = embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return vectors.astype('float32')

def embed_incident(incident: IncidentCandidate) -> np.ndarray:
    text = incident_to_text(incident)
    vec = embed_texts([text], batch_size=1)
    return vec[0]

sample_vec = embed_incident(incident_candidates[0]) if incident_candidates else np.array([])
sample_vec.shape


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(384,)

## Section 8 – Build Retrieval Corpus

We index three groups of documents:

- Current incident candidates.
- Synthetic historical incidents.
- Runbooks.


In [8]:
corpus_records = []

for inc, text in zip(incident_candidates, incident_texts):
    corpus_records.append({
        'doc_id': inc.incident_id,
        'doc_type': 'incident_candidate',
        'service': ','.join(inc.services),
        'title': f'Incident candidate {inc.incident_id}',
        'text': text,
    })

corpus_records.extend(historical_incidents)
corpus_records.extend(runbooks)

corpus_df = pd.DataFrame(corpus_records)
corpus_df.head()


,doc_id,doc_type,service,title,text
0,inc-001,incident_candidate,catalog-api,Incident candidate inc-001,incident_id=inc-001; services=catalog-api; ano...
1,inc-002,incident_candidate,catalog-api,Incident candidate inc-002,incident_id=inc-002; services=catalog-api; ano...
2,inc-003,incident_candidate,catalog-api,Incident candidate inc-003,incident_id=inc-003; services=catalog-api; ano...
3,inc-004,incident_candidate,catalog-api,Incident candidate inc-004,incident_id=inc-004; services=catalog-api; ano...
4,hist-001,historical_incident,checkout-api,Checkout latency spike after deployment,checkout-api experienced elevated latency and ...


## Section 9 – Build FAISS Index

We use an inner-product FAISS index on normalized embeddings, which is equivalent to cosine similarity.


In [9]:
corpus_vectors = embed_texts(corpus_df['text'].tolist(), batch_size=16)
embedding_dim = corpus_vectors.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(corpus_vectors)

print('Index size:', index.ntotal)
print('Embedding dim:', embedding_dim)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Index size: 10
Embedding dim: 384


## Section 10 – Add/Search Operations

Provide simple helper APIs so later notebooks can search the vector store by incident or free-text query.


In [10]:
def add_documents(new_docs: List[Dict[str, Any]]):
    global corpus_df, corpus_vectors, index
    if not new_docs:
        return
    new_df = pd.DataFrame(new_docs)
    new_vecs = embed_texts(new_df['text'].tolist(), batch_size=16)
    index.add(new_vecs)
    corpus_df = pd.concat([corpus_df, new_df], ignore_index=True)
    corpus_vectors = np.vstack([corpus_vectors, new_vecs])

def search_similar_by_text(query_text: str, top_k: int = 5) -> pd.DataFrame:
    q = embed_texts([query_text], batch_size=1)
    scores, ids = index.search(q, top_k)
    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        row = corpus_df.iloc[int(idx)].to_dict()
        row['score'] = float(score)
        rows.append(row)
    return pd.DataFrame(rows)

def search_similar_incidents(incident: IncidentCandidate, top_k: int = 5) -> pd.DataFrame:
    query_text = incident_to_text(incident)
    return search_similar_by_text(query_text, top_k=top_k)


## Section 11 – Retrieval Smoke Tests

Run similarity search for one incident candidate and one free-text operational query.


In [11]:
if incident_candidates:
    test_results = search_similar_incidents(incident_candidates[0], top_k=5)
    display(test_results[['doc_id', 'doc_type', 'service', 'title', 'score']])
else:
    print('No incident candidates available for retrieval test')


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,doc_id,doc_type,service,title,score
0,inc-001,incident_candidate,catalog-api,Incident candidate inc-001,1.000000
1,inc-003,incident_candidate,catalog-api,Incident candidate inc-003,0.996777
2,inc-002,incident_candidate,catalog-api,Incident candidate inc-002,0.993600
3,inc-004,incident_candidate,catalog-api,Incident candidate inc-004,0.974316
4,rb-002,runbook,generic,Database connection pool exhaustion remediation,0.353567


In [12]:
query = 'database pool exhaustion causing payment errors and high latency'
query_results = search_similar_by_text(query, top_k=5)
display(query_results[['doc_id', 'doc_type', 'service', 'title', 'score']])


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,doc_id,doc_type,service,title,score
0,hist-002,historical_incident,payments-service,Payment error spike due to DB pool exhaustion,0.543341
1,rb-002,runbook,generic,Database connection pool exhaustion remediation,0.388190
2,rb-001,runbook,generic,Investigate latency regression after deploy,0.387919
3,rb-003,runbook,generic,Rollback bad configuration change,0.337874
4,hist-001,historical_incident,checkout-api,Checkout latency spike after deployment,0.314232


## Section 12 – Persist Index Artifacts

Save the FAISS index and metadata so later notebooks can load them directly without recomputing embeddings.


In [13]:
index_path = vector_dir / 'incident_runbook.index'
meta_path = vector_dir / 'incident_runbook_metadata.parquet'

faiss.write_index(index, str(index_path))
corpus_df.to_parquet(meta_path, index=False)

print('Saved FAISS index to:', index_path)
print('Saved metadata to:', meta_path)


Saved FAISS index to: /workspace/agents026/data/vectorstore/incident_runbook.index
Saved metadata to: /workspace/agents026/data/vectorstore/incident_runbook_metadata.parquet


## Section 13 – Optional: Add One More Runbook Dynamically

This demonstrates the `add_documents` path, which can be useful later if you generate more runbooks or postmortem summaries on the fly.


In [14]:
extra_docs = [
    {
        'doc_id': 'rb-004',
        'doc_type': 'runbook',
        'service': 'generic',
        'title': 'Pod restart storm investigation',
        'text': 'Inspect pod restart counts, recent image versions, crash reasons, resource limits, and dependency availability. Avoid repeated blind restarts before confirming the underlying trigger.',
    }
]

add_documents(extra_docs)
print('Index size after add:', index.ntotal)

search_similar_by_text('pods restarting after rollout with crashloop', top_k=3)[['doc_id', 'doc_type', 'title', 'score']]


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Index size after add: 11


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,doc_id,doc_type,title,score
0,rb-004,runbook,Pod restart storm investigation,0.627157
1,hist-002,historical_incident,Payment error spike due to DB pool exhaustion,0.492087
2,rb-002,runbook,Database connection pool exhaustion remediation,0.468675
